# Preprocessing WDI Education Data

Notebook này làm preprocessing cho dữ liệu trong `data/Data.csv` theo hướng tạo **một file duy nhất cho dashboard**.

Các bước chính:

- Đọc dữ liệu gốc.
- Chuyển giá trị thiếu `..` thành `NaN`.
- Xóa dòng rỗng và footer của World Bank ở cuối file.
- Dùng bản wide-format nội bộ để tính feature, nhưng không xuất thêm file wide.
- Tạo thêm các feature quan trọng cho dashboard: nhóm chỉ số, cấp học, giới tính, đơn vị đo, số năm có dữ liệu, giá trị mới nhất, thay đổi theo thời gian, YoY, decade và giai đoạn 5 năm.
- Xuất 1 file CSV long-format enriched để FE dùng trực tiếp cho dashboard.

## 1. Import thư viện và khai báo đường dẫn

In [13]:
from pathlib import Path
import re

import pandas as pd

In [14]:
ROOT_DIR = Path.cwd()

# Nếu đang chạy notebook từ thư mục notebooks/, quay về thư mục gốc project.
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

RAW_DATA_PATH = ROOT_DIR / "data" / "Data.csv"
OUTPUT_DIR = ROOT_DIR / "data" / "processed"
LONG_OUTPUT_PATH = OUTPUT_DIR / "wdi_education_preprocessed.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH

WindowsPath('c:/Users/huynh/Documents/SOS/MATERIAL/TQHDL/lab2-wdi-education-dashboard/data/Data.csv')

## 2. Đọc dữ liệu

In [15]:
missing_values = ["..", "", " "]

df = pd.read_csv(RAW_DATA_PATH, na_values=missing_values)

df.shape

(192, 29)

In [16]:
df.head()

,Country Name,Country Code,Series Name,Series Code,2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],2009 [YR2009],2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Brunei Darussalam,BRN,"School enrollment, primary (% gross)",SE.PRM.ENRR,110.135422,112.115311,116.926270,119.105339,119.844757,121.505966,122.321259,116.783127,112.532433,108.705807,104.830895,103.310538,103.895142,103.847710,104.185795,105.726377,104.567054,104.179996,103.033009,102.709634,101.387927,99.134913,96.929219,94.820787,NaN
1,Brunei Darussalam,BRN,"School enrollment, secondary (% gross)",SE.SEC.ENRR,84.982307,86.768929,87.804131,90.892830,93.687309,97.559258,98.340858,98.772362,100.720039,101.011688,102.111122,105.229759,102.392693,96.284866,92.816170,90.098618,88.681473,89.181808,88.291031,86.685781,NaN,NaN,91.838154,NaN,NaN
2,Brunei Darussalam,BRN,"School enrollment, tertiary (% gross)",SE.TER.ENRR,14.668410,14.115470,14.257490,15.170780,15.249860,15.206880,15.484700,15.738510,16.565849,15.333160,17.424459,22.073351,23.464920,30.079910,28.664129,28.664410,31.243391,27.928599,27.771502,27.469256,27.425894,31.192537,36.418273,NaN,NaN
3,Brunei Darussalam,BRN,"Primary completion rate, total (% of relevant ...",SE.PRM.CMPT.ZS,122.554916,119.120522,121.753967,115.231392,113.224777,115.946068,118.802338,115.731392,110.637650,111.762082,111.028994,103.199446,103.014509,97.010251,98.826257,102.815448,103.848494,106.125040,100.955519,104.655998,NaN,NaN,101.681320,99.379915,NaN
4,Brunei Darussalam,BRN,"Lower secondary completion rate, total (% of r...",SE.SEC.CMPT.LO.ZS,NaN,NaN,NaN,103.541946,102.036842,103.717422,105.666870,105.062149,119.488251,109.028084,104.513236,106.267847,107.734960,101.072755,100.434722,97.955707,98.858876,100.982837,103.043149,108.190750,NaN,NaN,99.472675,104.975767,NaN


## 3. Xóa dòng rỗng và footer

File WDI thường có vài dòng rỗng và dòng mô tả như `Data from database...`, `Last Updated...` ở cuối file. Các dòng dữ liệu thật có `Country Code` dạng 3 chữ cái, ví dụ `VNM`, `KHM`, `BRN`.

In [17]:
df = df.dropna(how="all")

valid_country_code = df["Country Code"].astype("string").str.fullmatch(r"[A-Z]{3}", na=False)
df = df[valid_country_code]

df = df[df["Series Code"].notna()]

df.shape

(187, 29)

## 4. Chuẩn hóa kiểu dữ liệu

In [18]:
id_columns = ["Country Name", "Country Code", "Series Name", "Series Code"]
year_columns = [col for col in df.columns if re.fullmatch(r"\d{4} \[YR\d{4}\]", col)]

for col in id_columns:
    df[col] = df[col].astype("string").str.strip()

for col in year_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

year_columns[:3], year_columns[-3:]

(['2001 [YR2001]', '2002 [YR2002]', '2003 [YR2003]'],
 ['2023 [YR2023]', '2024 [YR2024]', '2025 [YR2025]'])

## 5. Feature engineering

Ta dùng dữ liệu wide-format nội bộ để tính các feature phục vụ dashboard, như nhóm chỉ số, số năm có dữ liệu, giá trị mới nhất và xu hướng tổng quát.

In [19]:
def classify_indicator(series_name):
    name = str(series_name).lower()

    if "expenditure" in name:
        category = "education_finance"
    elif "gdp" in name:
        category = "economy"
    elif "life expectancy" in name:
        category = "health"
    elif "population" in name and "urban" not in name:
        category = "demography"
    elif "urban population" in name:
        category = "urbanization"
    elif "literacy" in name:
        category = "literacy"
    elif "completion" in name:
        category = "completion"
    elif "out of school" in name:
        category = "out_of_school"
    elif "enrollment" in name:
        category = "enrollment"
    else:
        category = "other"

    if "primary and secondary" in name:
        level = "primary_secondary"
    elif "lower secondary" in name:
        level = "lower_secondary"
    elif "secondary" in name:
        level = "secondary"
    elif "primary" in name:
        level = "primary"
    elif "tertiary" in name:
        level = "tertiary"
    elif "adult" in name:
        level = "adult"
    else:
        level = "not_applicable"

    if "female" in name:
        gender = "female"
    elif "male" in name:
        gender = "male"
    else:
        gender = "total"

    if "gender parity index" in name or "(gpi)" in name:
        unit = "index"
    elif "% of" in name or "(%" in name:
        unit = "percent"
    elif "current us$" in name:
        unit = "current_usd"
    elif "years" in name:
        unit = "years"
    elif "population" in name or "children out of school" in name:
        unit = "people"
    else:
        unit = "value"

    return pd.Series({
        "indicator_category": category,
        "education_level": level,
        "gender": gender,
        "metric_unit": unit,
    })


def trend_direction(change):
    if pd.isna(change):
        return "unknown"
    if change > 0:
        return "up"
    if change < 0:
        return "down"
    return "flat"


year_lookup = {col: int(re.search(r"\d{4}", col).group()) for col in year_columns}
feature_df = df["Series Name"].apply(classify_indicator)
year_stats = df[year_columns]

available_year_count = year_stats.notna().sum(axis=1)
missing_year_count = year_stats.isna().sum(axis=1)

first_year = []
latest_year = []
first_value = []
latest_value = []

for _, row in year_stats.iterrows():
    valid_values = row.dropna()
    if valid_values.empty:
        first_year.append(pd.NA)
        latest_year.append(pd.NA)
        first_value.append(pd.NA)
        latest_value.append(pd.NA)
        continue

    first_col = valid_values.index[0]
    latest_col = valid_values.index[-1]
    first_year.append(year_lookup[first_col])
    latest_year.append(year_lookup[latest_col])
    first_value.append(valid_values.iloc[0])
    latest_value.append(valid_values.iloc[-1])

wide_df = pd.concat([df.reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)
wide_df["available_year_count"] = available_year_count.to_numpy()
wide_df["missing_year_count"] = missing_year_count.to_numpy()
wide_df["first_year_with_value"] = first_year
wide_df["latest_year_with_value"] = latest_year
wide_df["first_value"] = first_value
wide_df["latest_value"] = latest_value
wide_df["min_value"] = year_stats.min(axis=1).to_numpy()
wide_df["max_value"] = year_stats.max(axis=1).to_numpy()
wide_df["mean_value"] = year_stats.mean(axis=1).to_numpy()
wide_df["change_since_first"] = wide_df["latest_value"] - wide_df["first_value"]
wide_df["pct_change_since_first"] = wide_df["change_since_first"] / wide_df["first_value"].replace(0, pd.NA) * 100
wide_df["trend_direction"] = wide_df["change_since_first"].apply(trend_direction)

wide_df.head()

,Country Name,Country Code,Series Name,Series Code,year,value
0,Brunei Darussalam,BRN,"School enrollment, primary (% gross)",SE.PRM.ENRR,2001,110.135422
1,Brunei Darussalam,BRN,"School enrollment, secondary (% gross)",SE.SEC.ENRR,2001,84.982307
2,Brunei Darussalam,BRN,"School enrollment, tertiary (% gross)",SE.TER.ENRR,2001,14.668410
3,Brunei Darussalam,BRN,"Primary completion rate, total (% of relevant ...",SE.PRM.CMPT.ZS,2001,122.554916
4,Brunei Darussalam,BRN,"Lower secondary completion rate, total (% of r...",SE.SEC.CMPT.LO.ZS,2001,NaN


## 6. Chuyển sang long-format cho visual

Long-format giúp vẽ line chart, heatmap, ranking và filter theo năm dễ hơn. Đây là file duy nhất được xuất ra cho dashboard.

In [20]:
long_df = df.melt(
    id_vars=id_columns,
    value_vars=year_columns,
    var_name="year_column",
    value_name="value",
)
long_df["year"] = long_df["year_column"].str.extract(r"(\d{4})").astype(int)

long_df = long_df.rename(
    columns={
        "Country Name": "country_name",
        "Country Code": "country_code",
        "Series Name": "series_name",
        "Series Code": "series_code",
    }
)

long_df = long_df.merge(
    wide_df[
        [
            "Country Code",
            "Series Code",
            "indicator_category",
            "education_level",
            "gender",
            "metric_unit",
            "available_year_count",
            "missing_year_count",
            "first_year_with_value",
            "latest_year_with_value",
            "first_value",
            "latest_value",
            "change_since_first",
            "pct_change_since_first",
            "trend_direction",
        ]
    ].rename(columns={"Country Code": "country_code", "Series Code": "series_code"}),
    on=["country_code", "series_code"],
    how="left",
)

long_df = long_df.sort_values(["country_code", "series_code", "year"]).reset_index(drop=True)
group_cols = ["country_code", "series_code"]
long_df["previous_value"] = long_df.groupby(group_cols)["value"].shift(1)
long_df["yoy_change"] = long_df["value"] - long_df["previous_value"]
long_df["yoy_change_pct"] = long_df["yoy_change"] / long_df["previous_value"].replace(0, pd.NA) * 100
long_df["is_value_available"] = long_df["value"].notna()
long_df["is_latest_year_with_value"] = long_df["latest_year_with_value"].notna() & long_df["year"].eq(long_df["latest_year_with_value"].astype("Int64"))
long_df["decade"] = (long_df["year"] // 10 * 10).astype(str) + "s"
long_df["period_5y"] = ((long_df["year"] - 2001) // 5 * 5 + 2001).astype(str) + "-" + (((long_df["year"] - 2001) // 5 * 5 + 2005).astype(str))

long_df.head()

,country_name,country_code,series_name,series_code,year,value
0,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2001,18287.827228
1,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2002,18621.292258
2,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2003,20677.900114
3,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2004,24423.094700
4,Brunei Darussalam,BRN,GDP per capita (current US$),NY.GDP.PCAP.CD,2005,29386.270382


## 7. Kiểm tra nhanh dữ liệu sau preprocessing

In [21]:
print("Dashboard rows, columns:", long_df.shape)
print("Countries:", long_df["country_code"].nunique())
print("Indicators:", long_df["series_code"].nunique())
print("Years:", long_df["year"].min(), "-", long_df["year"].max())
print("Missing values in value:", long_df["value"].isna().sum())
print("Feature columns:", [col for col in long_df.columns if col not in ["country_name", "country_code", "series_name", "series_code", "year", "value"]])

Rows, columns: (4675, 6)
Countries: 11
Indicators: 17
Years: 2001 - 2025
Missing values in value: 1466


In [22]:
duplicate_count = long_df.duplicated(["country_code", "series_code", "year"]).sum()
duplicate_count

np.int64(0)

## 8. Xuất CSV đã xử lý

- `wdi_education_preprocessed.csv`: long-format enriched, file duy nhất cho dashboard visual.

In [23]:
long_df.to_csv(LONG_OUTPUT_PATH, index=False, encoding="utf-8-sig")

LONG_OUTPUT_PATH

WindowsPath('c:/Users/huynh/Documents/SOS/MATERIAL/TQHDL/lab2-wdi-education-dashboard/data/processed/wdi_education_preprocessed.csv')